# 7 · AI assistance with Ollama and RustGIS

**Question:** how can an assistant propose map operations while the analyst keeps control of the data and commands?

RustGIS has its **own AI Assistant**. Its [assistant guide](https://github.com/opengeos/RustGIS/blob/main/docs/user-guide/assistant.md) documents Ollama as a local provider, usually at `http://localhost:11434`. This notebook teaches an inspectable plan; it does not pretend to invoke RustGIS from Pyodide or send data to a model.

![Where RustGIS and Ollama run](https://jltobias.github.io/JupyterLite-RustGIS/assets/runtime-boundaries.svg)

## Local setup, outside JupyterLite

1. Install and start [Ollama](https://ollama.com/). Pull a tool-capable model, such as the RustGIS guide's `qwen3` example.
2. In **native RustGIS**, open **Analyze → AI Assistant**, choose **Ollama**, and select the model.
3. In RustGIS Web, browser access to your local server needs an allowed origin. The RustGIS guide gives `OLLAMA_ORIGINS=https://web.rustgis.app ollama serve`. Browser/network policies may still block it.
4. Inspect every command and its result in the RustGIS conversation. Commands with file, network, plugin, script, or project replacement effects require explicit RustGIS approval.

## Other assistant providers

The [RustGIS guide](https://github.com/opengeos/RustGIS/blob/main/docs/user-guide/assistant.md) also documents Anthropic, OpenAI, Gemini, OpenRouter, and OpenAI-compatible local servers such as LM Studio. Provider choice changes where prompts and any read map data are sent. For this lesson, Ollama is the local example; no provider is contacted by the notebook.

## Author a bounded question

The prompt names the dataset and asks for a plan first. It avoids passing fabricated heights as real evidence.

In [ ]:
question = (
    'Using the illustrative study footprints layer, propose a way to symbolize height_m. '
    'State that heights are invented teaching values. List the fields you would inspect '
    'and the RustGIS commands you would use, but do not run anything yet.'
)
print(question)

## A typed, authored example response

This fixture is **not** model output. It lets us practice validation even when Ollama is absent.

In [ ]:
proposal = {
    'layer':'Illustrative study footprints',
    'field':'height_m',
    'symbol':'graduated',
    'classes':3,
    'minimum':0,
    'maximum':100,
    'note':'Illustrative heights only; do not infer real building height.'
}
print(proposal)

## Validate before acting

The notebook accepts only a small set of values. It does not execute generated Python, SQL, shell, URLs, or RustGIS commands.

In [ ]:
def validate_plan(plan, fields):
    if plan.get('field') not in fields: raise ValueError('Unknown field')
    if plan.get('symbol') not in {'graduated','single'}: raise ValueError('Unsupported symbol')
    if type(plan.get('classes')) is not int or not 2 <= plan['classes'] <= 7: raise ValueError('Invalid class count')
    if not 0 <= plan.get('minimum',-1) < plan.get('maximum',-1) <= 500: raise ValueError('Invalid range')
    return True
assert validate_plan(proposal, {'name','kind','height_m'})
print('Plan passes the teaching checks; a person still reviews it.')

## Visualize the proposal without changing a GIS project

In [ ]:
from IPython.display import display, HTML
from viz import bars
display(HTML(bars(['Low','Medium','High'], [33,67,100], 'Illustrative graduated class ceilings', ' m', '#e9a461')))

## Reject an unsafe or unsupported proposal

In [ ]:
bad = dict(proposal, field='download_and_run', classes=100)
try: validate_plan(bad, {'name','kind','height_m'})
except ValueError as exc: print('Rejected:',exc)

## Try the actual RustGIS assistant

Open [RustGIS Web with this layer](https://web.rustgis.app/?blank&data=https%3A%2F%2Fjltobias.github.io%2FJupyterLite-RustGIS%2Fdata%2Fdemo_features.geojson) or use desktop RustGIS. Open the AI Assistant pane. Ask the question above. Review its proposed command parameters and output, and use undo if a map change is wrong. The upstream guide says provider requests can include layer names, fields, extent, and rows the assistant reads, so choose an appropriate model/provider for your data.

## Optional native CLI

`rustgis-cli ask --sample "Which 5 countries have the largest GDP?"` is the upstream quick-start example. Run it in a local terminal with a configured assistant provider; it is not a JupyterLite command.

**Reflection:** Which parts of the assistant response are observations about the data, which are proposed operations, and which require your approval or independent verification?

**Citations:** [RustGIS AI Assistant guide](https://github.com/opengeos/RustGIS/blob/main/docs/user-guide/assistant.md), [RustGIS CLI guide](https://github.com/opengeos/RustGIS/blob/main/docs/cli.md), [Ollama](https://ollama.com/), [your GeoLibre–Ollama teaching notebooks](https://github.com/jltobias/JupyterLite-GeoLibre-GeoAI/tree/main/book/notebooks).